# Nighttime Lens Flare Removal — Final Working Notebook

**What changed compared with the earlier notebook (and why)**

| Problem found | Fix used here |
|---|---|
| Flares were not removed: the small U-Net trained from scratch only learned to copy its input | **ImageNet-pretrained ResNet-34 encoder** (large receptive field for big glows, converges far faster) |
| Network had to rebuild the whole image from a grey sigmoid output | **Residual output:** `output = input - predicted_flare` (starts as identity) |
| Training took ~14 min/epoch because the CPU generated flares on the fly | **Data cached once in RAM** (6000 train / 300 val pairs, flips as augmentation) → GPU-speed training |
| Training pairs were made at 256px, so the official loader's 1440px→256 **center crop** showed only a zoomed-in patch of glow (streaks/light source often cut off) | **Pairs generated at 512px (official size), random 256 crops for training**, validation on full 512 images |
| Plain L1 barely penalises thin streaks (few pixels) and training stopped after 24 epochs (~25 of 90 min) | **Flare-weighted L1** (x5 on flare pixels) and **up to 70 epochs** within an 80-min budget per model |
| Constant low LR, few steps | **OneCycle LR schedule**, AdamW, mixed precision, ~20k steps per model |
| Variables / state lost on disconnect | **Resumable** checkpoints, chunked cache, a time budget per model |

**Models (identical training for a fair comparison)**
1. **Baseline:** ResNet-34 U-Net.
2. **Proposed:** same network + attention gates on the skip connections + multi-scale dilated/global-context bottleneck.

**Run top to bottom.** Needs a GPU (Kaggle: Settings → Accelerator → GPU) and Internet ON (clones Flare7K, downloads ImageNet weights).
Turn the GPU off in Kaggle when you are not training, it counts against the weekly quota.

In [ ]:
# Cell 1 — Imports, settings, GPU check
import os, sys, json, math, time, copy, random, re, shutil, subprocess, warnings
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import matplotlib.pyplot as plt
import torchvision, torchvision.transforms as T
warnings.filterwarnings("ignore")

# ---------------- settings you may change ----------------
SEED            = 42
GEN_SIZE        = 512           # pairs generated at the official Flare7K++ size (flare scale matches the 512px test set)
CROP_SIZE       = 256           # random training crop taken from each 512 pair
BATCH_SIZE      = 8
LR              = 3e-4
EPOCHS          = 120           # planned epochs per model (auto-reduced if TIME_BUDGET_MIN is too small)
TIME_BUDGET_MIN = 120           # max training minutes PER MODEL (protects your weekly GPU quota)
N_TRAIN_CACHE   = 5000          # cached training pairs (uint8, 512px: ~7.9 GB RAM)
N_VAL_CACHE     = 300           # cached validation pairs
SSIM_WEIGHT     = 0.1
FLARE_WEIGHT    = 4.0           # extra L1 weight on flare pixels (thin streaks are a tiny fraction of the image)
TRAIN_BASELINE  = True
TRAIN_PROPOSED  = True
NUM_WORKERS     = min(4, os.cpu_count() or 1)   # only used while building the cache
# ----------------------------------------------------------

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)
if device.type != "cuda":
    print("WARNING: no GPU. Enable it in Kaggle Settings -> Accelerator before training (CPU is ~20x slower).")
else:
    print("GPU:", torch.cuda.get_device_name(0))
AMP = device.type == "cuda"

In [ ]:
# Cell 2 — Locate the Kaggle dataset and define all paths
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/fidasaifudheen/nighttime-flare-removal-data"),
    Path("/kaggle/input/nighttime-flare-removal-data"),
    *Path("/kaggle/input/datasets").glob("*/nighttime-flare-removal-data"),
    *Path("/kaggle/input").glob("*/nighttime-flare-removal-data"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError("Attach the Kaggle dataset 'nighttime-flare-removal-data' (Add Input).")

OUTPUT_ROOT = Path("/kaggle/working/Flare_Final")
CACHE_DIR   = Path("/kaggle/temp"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
for d in ["checkpoints", "results/figures", "results/metrics"]:
    (OUTPUT_ROOT / d).mkdir(parents=True, exist_ok=True)

FLARE7KPP_ROOT          = DATA_ROOT / "Flare7K" / "Flare7Kpp"
BACKGROUND_DIR          = DATA_ROOT / "Flickr24K" / "Flickr24K"
SCATTERING_ROOT         = FLARE7KPP_ROOT / "Flare7K" / "Scattering_Flare"
SCATTERING_COMPOUND_DIR = SCATTERING_ROOT / "Compound_Flare"
SCATTERING_LIGHT_DIR    = SCATTERING_ROOT / "Light_Source"
REFLECTIVE_DIR          = FLARE7KPP_ROOT / "Flare7K" / "Reflective_Flare"
FLARE_R_ROOT            = FLARE7KPP_ROOT / "Flare-R"
FLARE_R_COMPOUND_DIR    = FLARE_R_ROOT / "Compound_Flare"
FLARE_R_LIGHT_DIR       = FLARE_R_ROOT / "Light_Source"
REAL_INPUT_DIR  = FLARE7KPP_ROOT / "test_data" / "real" / "input"
REAL_GT_DIR     = FLARE7KPP_ROOT / "test_data" / "real" / "gt"
SYNTH_INPUT_DIR = FLARE7KPP_ROOT / "test_data" / "synthetic" / "input"
SYNTH_GT_DIR    = FLARE7KPP_ROOT / "test_data" / "synthetic" / "gt"

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
def image_files(folder):
    return sorted([p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

for name, folder in {"Background": BACKGROUND_DIR, "Scattering": SCATTERING_COMPOUND_DIR,
                     "Scattering light": SCATTERING_LIGHT_DIR, "Reflective": REFLECTIVE_DIR,
                     "Flare-R": FLARE_R_COMPOUND_DIR, "Flare-R light": FLARE_R_LIGHT_DIR,
                     "Real input": REAL_INPUT_DIR, "Real gt": REAL_GT_DIR,
                     "Synthetic input": SYNTH_INPUT_DIR, "Synthetic gt": SYNTH_GT_DIR}.items():
    assert folder.is_dir(), f"Missing {name}: {folder}"
    print(f"{name:18s}: {len(image_files(folder)):6d} images")
print("Output dir:", OUTPUT_ROOT)

In [ ]:
# Cell 3 — Official Flare7K++ paired loader (source-preserving targets)
OFFICIAL_REPO = Path("/kaggle/working/Flare7K")
if not OFFICIAL_REPO.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ykdai/Flare7K.git", str(OFFICIAL_REPO)], check=True)
if str(OFFICIAL_REPO) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_REPO))
from basicsr.data.flare7kpp_dataset import Flare7kpp_Pair_Loader

official_opt = {
    "image_path": str(BACKGROUND_DIR),
    "transform_base":  {"img_size": GEN_SIZE},
    "transform_flare": {"scale_min": 0.8, "scale_max": 1.5, "translate": 300, "shear": 20},
    "mask_type": None,
    "scattering_dict": {"Flare7K_scattering": str(SCATTERING_COMPOUND_DIR),
                        "Flare_R_scattering": str(FLARE_R_COMPOUND_DIR)},
    "reflective_dict": {"Flare7K_reflective": str(REFLECTIVE_DIR), "Flare_R_reflective": None},
    "light_dict": {"Flare7K_light": str(SCATTERING_LIGHT_DIR), "Flare_R_light": str(FLARE_R_LIGHT_DIR)},
    "data_ratio": [0.5, 0.5],
}
full_official_loader = Flare7kpp_Pair_Loader(official_opt)
assert full_official_loader.light_flag, "Light-source pairing is off; check the light_dict paths."

s = full_official_loader[0]
print({k: tuple(v.shape) for k, v in s.items() if hasattr(v, "shape")})
assert s["lq"].shape == (3, GEN_SIZE, GEN_SIZE) and s["gt"].shape == s["lq"].shape
print("Official loader OK. Backgrounds:", len(full_official_loader.data_list))

In [ ]:
# Cell 4 — Leakage-safe split (backgrounds, flares, light sources, reflective flares) + wrapper dataset
VAL_FRACTION = 0.20
def split_items(items, seed, val_fraction=VAL_FRACTION):
    items = sorted(list(items)); rng = random.Random(seed); rng.shuffle(items)
    n_val = max(1, int(len(items) * val_fraction)) if len(items) > 1 else 0
    return items[n_val:], items[:n_val]

def split_flare_light_pairs(flare_paths, light_paths, seed):
    fm = {Path(p).name: p for p in flare_paths}; lm = {Path(p).name: p for p in light_paths}
    if set(fm) != set(lm): raise ValueError("Flare/light filenames do not match")
    tr, va = split_items(sorted(fm), seed)
    return [fm[n] for n in tr], [lm[n] for n in tr], [fm[n] for n in va], [lm[n] for n in va]

tr_bg, va_bg = split_items(full_official_loader.data_list, SEED)
trF, trL, vaF, vaL = [], [], [], []
for gi, (fg, lg) in enumerate(zip(full_official_loader.flare_list, full_official_loader.light_list)):
    a, b, c, d = split_flare_light_pairs(fg, lg, 100 + gi)
    trF.append(a); trL.append(b); vaF.append(c); vaL.append(d)
trR, vaR = [], []
for gi, g in enumerate(full_official_loader.reflective_list):
    a, b = split_items(g, 200 + gi); trR.append(a); vaR.append(b)

train_official_loader = copy.deepcopy(full_official_loader)
val_official_loader   = copy.deepcopy(full_official_loader)
for ld, bg, fl, li, rf in [(train_official_loader, tr_bg, trF, trL, trR), (val_official_loader, va_bg, vaF, vaL, vaR)]:
    ld.data_list, ld.flare_list, ld.light_list, ld.reflective_list = bg, fl, li, rf
    ld.data_ratio = [0.5, 0.5]
assert set(tr_bg).isdisjoint(va_bg)
for a, b in zip(trF, vaF): assert set(a).isdisjoint(b)
print("Backgrounds train/val:", len(tr_bg), "/", len(va_bg), "| splits disjoint: OK")

class FlareRestorationDataset(Dataset):
    """official lq -> input, source-preserving gt -> target. deterministic=True gives repeatable samples."""
    def __init__(self, official_dataset, deterministic=False, seed=42000):
        self.ds, self.det, self.seed = official_dataset, deterministic, seed
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        if not self.det:
            s = self.ds[idx]
        else:
            ps, ns, ts = random.getstate(), np.random.get_state(), torch.get_rng_state()
            try:
                fs = self.seed + int(idx); random.seed(fs); np.random.seed(fs); torch.manual_seed(fs)
                s = self.ds[idx]
            finally:
                random.setstate(ps); np.random.set_state(ns); torch.set_rng_state(ts)
        x, y = s["lq"].float(), s["gt"].float()
        if not (torch.isfinite(x).all() and torch.isfinite(y).all()): raise ValueError(f"non-finite sample {idx}")
        return {"input": x, "target": y}

train_dataset = FlareRestorationDataset(train_official_loader, deterministic=False)
val_dataset   = FlareRestorationDataset(val_official_loader,   deterministic=True, seed=42000)
print("Train pairs available:", len(train_dataset), "| Val pairs available:", len(val_dataset))

In [ ]:
# Cell 5 — Build the in-memory cache ONCE (512px pairs stored as uint8; chunked + resumable; CPU work)
# Why 512: the official loader CENTER-CROPS each 1440px flare to img_size. At 256 the model only saw a
# zoomed-in patch of glow (light source / streaks often outside the crop), unlike the 512px test images.
def build_cache(ds, n, path, seed, chunk=500):
    path = str(path)
    if os.path.exists(path):
        print("cache exists:", path); return
    idx = torch.randperm(len(ds), generator=torch.Generator().manual_seed(seed))[:n].tolist()
    parts, t0 = [], time.time()
    for s in range(0, n, chunk):
        cp = f"{path}.part{s}"
        if not os.path.exists(cp):
            ld = DataLoader(Subset(ds, idx[s:s+chunk]), batch_size=16, shuffle=False, num_workers=NUM_WORKERS)
            X, Y = [], []
            for b in ld:
                X.append((b["input"].clamp(0, 1) * 255).round().to(torch.uint8))
                Y.append((b["target"].clamp(0, 1) * 255).round().to(torch.uint8))
            torch.save({"x": torch.cat(X), "y": torch.cat(Y)}, cp)
        parts.append(cp)
        print(f"{min(s+chunk, n)}/{n} | {(time.time()-t0)/60:.1f} min", flush=True)
    X = torch.empty((n, 3, GEN_SIZE, GEN_SIZE), dtype=torch.uint8); Y = torch.empty_like(X); k = 0
    for p in parts:                                   # assemble without doubling RAM
        d = torch.load(p); m = len(d["x"]); X[k:k+m] = d["x"]; Y[k:k+m] = d["y"]; k += m; del d
    torch.save({"x": X, "y": Y}, path); del X, Y
    for p in parts: os.remove(p)
    print("saved", path)

TRAIN_PT = CACHE_DIR / f"train{N_TRAIN_CACHE}_{GEN_SIZE}.pt"
VAL_PT   = CACHE_DIR / f"val{N_VAL_CACHE}_{GEN_SIZE}.pt"
build_cache(val_dataset,   N_VAL_CACHE,   VAL_PT,   seed=2)
build_cache(train_dataset, N_TRAIN_CACHE, TRAIN_PT, seed=1)


In [ ]:
# Cell 6 — Models: ResNet-34 U-Net (baseline) and the proposed attention + multi-scale version
from torchvision.models import resnet34
try:
    from torchvision.models import ResNet34_Weights
    _W = ResNet34_Weights.IMAGENET1K_V1
except Exception:
    _W = None

class ConvBNAct(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class AttentionGate(nn.Module):
    """Additive attention gate on a skip connection (decoder feature g, encoder skip x, same spatial size)."""
    def __init__(self, cg, cx):
        super().__init__()
        inter = max(cx // 2, 8)
        self.wg  = nn.Sequential(nn.Conv2d(cg, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.wx  = nn.Sequential(nn.Conv2d(cx, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.psi = nn.Sequential(nn.Conv2d(inter, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        return x * self.psi(F.relu(self.wg(g) + self.wx(x), inplace=True))

class MultiScaleBottleneck(nn.Module):
    """Parallel dilated convolutions + global-context branch, fused and added residually."""
    def __init__(self, c, dilations=(1, 2, 4)):
        super().__init__()
        q = c // 4
        self.branches = nn.ModuleList([nn.Sequential(
            nn.Conv2d(c, q, 3, padding=d, dilation=d, bias=False), nn.BatchNorm2d(q), nn.ReLU(inplace=True))
            for d in dilations])
        self.glob = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, q, 1), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(nn.Conv2d(q * (len(dilations) + 1), c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
    def forward(self, x):
        outs = [b(x) for b in self.branches]
        g = self.glob(x).expand(-1, -1, x.shape[2], x.shape[3])
        return x + self.fuse(torch.cat(outs + [g], 1))

class DecBlock(nn.Module):
    def __init__(self, cup, cskip, cout, attention):
        super().__init__()
        self.up   = nn.ConvTranspose2d(cup, cout, 2, stride=2)
        self.gate = AttentionGate(cout, cskip) if attention else None
        self.conv = nn.Sequential(ConvBNAct(cout + cskip, cout), ConvBNAct(cout, cout))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        if self.gate is not None: skip = self.gate(x, skip)
        return self.conv(torch.cat([x, skip], 1))

class FlareUNet(nn.Module):
    """Predicts a flare layer; output = clamp(input - flare, 0, 1). Input/output in [0,1]."""
    def __init__(self, attention=False, multiscale=False, pretrained=True):
        super().__init__()
        try:
            r = resnet34(weights=_W if pretrained else None)
        except Exception as e:
            print("WARNING: could not load ImageNet weights (", type(e).__name__, ") -> random init. Turn Internet ON.")
            r = resnet34(weights=None)
        self.stem, self.pool = nn.Sequential(r.conv1, r.bn1, r.relu), r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.bottleneck = MultiScaleBottleneck(512) if multiscale else nn.Identity()
        self.full = ConvBNAct(3, 32)
        self.d3 = DecBlock(512, 256, 256, attention)
        self.d2 = DecBlock(256, 128, 128, attention)
        self.d1 = DecBlock(128,  64,  64, attention)
        self.d0 = DecBlock( 64,  64,  64, attention)
        self.dF = DecBlock( 64,  32,  32, attention)
        self.head = nn.Conv2d(32, 3, 1)
        nn.init.normal_(self.head.weight, std=1e-3); nn.init.zeros_(self.head.bias)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, inp):
        x = (inp - self.mean) / self.std
        f0 = self.stem(x); f1 = self.l1(self.pool(f0)); f2 = self.l2(f1); f3 = self.l3(f2); f4 = self.l4(f3)
        b = self.bottleneck(f4)
        d = self.d3(b, f3); d = self.d2(d, f2); d = self.d1(d, f1); d = self.d0(d, f0)
        d = self.dF(d, self.full(x))
        flare = self.head(d).float()
        return torch.clamp(inp.float() - flare, 0.0, 1.0)

def build(name, pretrained=True):
    return {"baseline": lambda: FlareUNet(False, False, pretrained),
            "proposed": lambda: FlareUNet(True,  True,  pretrained)}[name]()
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

# smoke test
for n in ["baseline", "proposed"]:
    m = build(n, pretrained=False).to(device).eval()
    with torch.no_grad():
        o = m(torch.rand(2, 3, 256, 256, device=device))
    print(f"{n:9s} params {count_params(m)/1e6:5.1f}M | out {tuple(o.shape)} range ({o.min():.3f},{o.max():.3f})")
    del m

In [ ]:
# Cell 7 — Loss, metrics and a resumable training function
def _gauss(size=11, sigma=1.5, ch=3, dev="cpu"):
    g = torch.arange(size, dtype=torch.float32, device=dev) - size // 2
    g = torch.exp(-(g ** 2) / (2 * sigma ** 2)); g = g / g.sum()
    return (g[:, None] @ g[None, :])[None, None].expand(ch, 1, size, size).contiguous()

def ssim_per_image(x, y, win=11):
    C = x.shape[1]; w = _gauss(win, 1.5, C, x.device); p = win // 2
    mx, my = F.conv2d(x, w, padding=p, groups=C), F.conv2d(y, w, padding=p, groups=C)
    sxx = F.conv2d(x * x, w, padding=p, groups=C) - mx ** 2
    syy = F.conv2d(y * y, w, padding=p, groups=C) - my ** 2
    sxy = F.conv2d(x * y, w, padding=p, groups=C) - mx * my
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    s = ((2 * mx * my + c1) * (2 * sxy + c2)) / ((mx ** 2 + my ** 2 + c1) * (sxx + syy + c2))
    return s.flatten(1).mean(1)

def psnr_per_image(x, y):
    mse = ((x - y) ** 2).flatten(1).mean(1).clamp_min(1e-10)
    return 10 * torch.log10(1.0 / mse)

def criterion(pred, y, x):
    """Flare-weighted L1 + SSIM. Weight = 1 + FLARE_WEIGHT on pixels where input and target differ (the flare),
    normalised so the loss scale matches plain L1 on flare-free images."""
    with torch.no_grad():
        m = ((x - y).abs().amax(1, keepdim=True) / 0.05).clamp(0, 1)   # soft flare mask, ~1 where flare >= 0.05
        w = 1.0 + FLARE_WEIGHT * m
    l1 = (w * (pred - y).abs()).mean() / w.mean()
    return l1 + SSIM_WEIGHT * (1 - ssim_per_image(pred, y).mean())

class RAMPairs(Dataset):
    """uint8 512px pairs. augment=True: aligned random CROP_SIZE crop + flips/rot90. augment=False: full 512 image."""
    def __init__(self, path, augment):
        try: d = torch.load(str(path), mmap=True)
        except TypeError: d = torch.load(str(path))
        self.x, self.y, self.aug = d["x"], d["y"], augment
    def __len__(self): return len(self.x)
    def __getitem__(self, i):
        x, y = self.x[i], self.y[i]
        if self.aug:
            H, W = x.shape[-2:]
            t = int(torch.randint(0, H - CROP_SIZE + 1, (1,))); l = int(torch.randint(0, W - CROP_SIZE + 1, (1,)))
            x, y = x[:, t:t+CROP_SIZE, l:l+CROP_SIZE], y[:, t:t+CROP_SIZE, l:l+CROP_SIZE]
        x, y = x.float() / 255.0, y.float() / 255.0
        if self.aug:
            if torch.rand(1) < 0.5: x, y = x.flip(-1), y.flip(-1)
            if torch.rand(1) < 0.5: x, y = x.flip(-2), y.flip(-2)
            if torch.rand(1) < 0.5:  # rot90 keeps the pair aligned
                x, y = torch.rot90(x, 1, (1, 2)), torch.rot90(y, 1, (1, 2))
        return {"input": x, "target": y}

@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval(); ps, ss, l1 = [], [], []
    for i, b in enumerate(loader):
        if max_batches and i >= max_batches: break
        x, y = b["input"].to(device), b["target"].to(device)
        with torch.autocast(device_type=device.type, enabled=AMP):
            p = model(x)
        p = p.float().clamp(0, 1)
        ps.append(psnr_per_image(p, y)); ss.append(ssim_per_image(p, y)); l1.append((p - y).abs().flatten(1).mean(1))
    ps, ss, l1 = torch.cat(ps), torch.cat(ss), torch.cat(l1)
    return {"psnr": ps.mean().item(), "ssim": ss.mean().item(), "l1": l1.mean().item(), "n": len(ps)}

class Identity(nn.Module):
    def forward(self, x): return x

def fit_model(name, epochs=EPOCHS, time_budget_min=TIME_BUDGET_MIN):
    ckdir = OUTPUT_ROOT / "checkpoints" / name; ckdir.mkdir(parents=True, exist_ok=True)
    latest, best = ckdir / "latest.pth", ckdir / "best.pth"
    tr = DataLoader(RAMPairs(TRAIN_PT, True),  batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                    num_workers=0, pin_memory=(device.type == "cuda"))
    va = DataLoader(RAMPairs(VAL_PT, False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = build(name).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    start, hist, best_psnr, planned = 0, [], -1.0, epochs

    if latest.exists():                                  # resume
        ck = torch.load(latest, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start, hist, best_psnr, planned = ck["epoch"] + 1, ck["hist"], ck["best_psnr"], ck["planned"]
        print(f"[{name}] resumed from epoch {start}/{planned}")
    else:                                                # fit the schedule to the time budget
        tmp = copy.deepcopy(model).train(); topt = torch.optim.SGD(tmp.parameters(), lr=0.0)
        it = iter(tr); t0 = None
        for k in range(25):
            if k == 5:
                torch.cuda.synchronize() if device.type == "cuda" else None; t0 = time.time()
            try: b = next(it)
            except StopIteration: it = iter(tr); b = next(it)
            x, y = b["input"].to(device), b["target"].to(device)
            with torch.autocast(device_type=device.type, enabled=AMP):
                p = tmp(x)
            l = criterion(p.float(), y, x); topt.zero_grad(); l.backward(); topt.step()
        torch.cuda.synchronize() if device.type == "cuda" else None
        sec_step = (time.time() - t0) / 20; del tmp, topt
        sec_epoch = sec_step * len(tr) * 1.08
        planned = int(max(1, min(epochs, (time_budget_min * 60) // sec_epoch)))
        print(f"[{name}] {sec_step:.3f}s/step, ~{sec_epoch/60:.1f} min/epoch -> training {planned} epochs "
              f"(~{planned*sec_epoch/60:.0f} min)")
    total = planned * len(tr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=total, pct_start=0.1,
                                                anneal_strategy="cos", div_factor=20, final_div_factor=200)
    if latest.exists() and start > 0:
        sched.load_state_dict(ck["sched"])

    for ep in range(start, planned):
        model.train(); t0 = time.time(); run = n = 0
        for b in tr:
            x, y = b["input"].to(device, non_blocking=True), b["target"].to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=AMP):
                p = model(x)
            loss = criterion(p.float(), y, x)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item() * x.size(0); n += x.size(0)
        v = evaluate(model, va)
        rec = {"epoch": ep, "train_loss": run / n, "val_l1": v["l1"], "val_psnr": v["psnr"], "val_ssim": v["ssim"],
               "sec": time.time() - t0}
        hist.append(rec)
        print(f"[{name}] epoch {ep+1}/{planned} | train {rec['train_loss']:.4f} | val L1 {v['l1']:.4f} "
              f"PSNR {v['psnr']:.2f} SSIM {v['ssim']:.4f} | {rec['sec']/60:.1f} min")
        ck = {"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
              "epoch": ep, "hist": hist, "best_psnr": best_psnr, "planned": planned}
        if v["psnr"] > best_psnr:
            best_psnr = v["psnr"]; ck["best_psnr"] = best_psnr; torch.save({"model": model.state_dict()}, best)
        torch.save(ck, latest)
    json.dump(hist, open(ckdir / "history.json", "w"), indent=1)
    print(f"[{name}] done. best val PSNR {best_psnr:.2f}")
    return model, hist

va_ref = DataLoader(RAMPairs(VAL_PT, False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
ref = evaluate(Identity().to(device), va_ref)
print("Reference on validation cache, input left untouched:", {k: round(v, 4) for k, v in ref.items()})
print("A trained model must beat this PSNR/L1 to be removing flares.")

In [ ]:
# Cell 8 — Train the BASELINE (resumable; re-run the cell after a disconnect to continue)
if TRAIN_BASELINE:
    base_model, base_hist = fit_model("baseline")
else:
    print("skipped")

In [ ]:
# Cell 9 — Train the PROPOSED model (same data, same schedule)
if TRAIN_PROPOSED:
    prop_model, prop_hist = fit_model("proposed")
else:
    print("skipped")

In [ ]:
# Cell 10 — Training curves
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for name in ["baseline", "proposed"]:
    p = OUTPUT_ROOT / "checkpoints" / name / "history.json"
    if not p.exists(): continue
    h = json.load(open(p)); e = [r["epoch"] + 1 for r in h]
    ax[0].plot(e, [r["train_loss"] for r in h], label=name)
    ax[1].plot(e, [r["val_psnr"] for r in h], label=name)
    ax[2].plot(e, [r["val_ssim"] for r in h], label=name)
ax[0].axhline(0, c="none")
for a, t in zip(ax, ["Train loss", "Validation PSNR (dB)", "Validation SSIM"]):
    a.set_title(t); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
ax[1].axhline(ref["psnr"], ls="--", c="gray"); ax[2].axhline(ref["ssim"], ls="--", c="gray")
plt.tight_layout(); plt.savefig(OUTPUT_ROOT / "results/figures/training_curves.png", dpi=140); plt.show()
print("Dashed grey line = input left untouched. Curves above it mean flares are being removed.")

In [ ]:
# Cell 11 — Final evaluation on the official REAL and SYNTHETIC test sets (512x512 full images)
def numeric_id(fn):
    m = re.search(r"(\d+)", fn); return m.group(1) if m else Path(fn).stem

class PairedFolderDataset(Dataset):
    def __init__(self, input_dir, target_dir):
        im = {numeric_id(p.name): p for p in image_files(input_dir)}
        tm = {numeric_id(p.name): p for p in image_files(target_dir)}
        assert set(im) == set(tm), "Input/GT ids differ"
        self.pairs = [(im[k], tm[k]) for k in sorted(im)]; self.tt = T.ToTensor()
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        a, b = self.pairs[i]
        with Image.open(a) as f: x = self.tt(f.convert("RGB"))
        with Image.open(b) as f: y = self.tt(f.convert("RGB"))
        assert x.shape == y.shape and x.shape[-1] % 32 == 0 and x.shape[-2] % 32 == 0, x.shape
        return {"input": x, "target": y, "filename": a.name}

test_sets = {"real":      PairedFolderDataset(REAL_INPUT_DIR, REAL_GT_DIR),
             "synthetic": PairedFolderDataset(SYNTH_INPUT_DIR, SYNTH_GT_DIR)}
def load_best(name):
    m = build(name, pretrained=False).to(device)
    m.load_state_dict(torch.load(OUTPUT_ROOT / "checkpoints" / name / "best.pth", map_location=device)["model"])
    return m.eval()

models = {"Input (no processing)": Identity().to(device).eval()}
for n, label in [("baseline", "Baseline ResNet34-UNet"), ("proposed", "Proposed (attn + multiscale)")]:
    if (OUTPUT_ROOT / "checkpoints" / n / "best.pth").exists():
        models[label] = load_best(n)

@torch.no_grad()
def time_model(m, size=(1, 3, 512, 512), runs=10):
    x = torch.rand(*size, device=device)
    for _ in range(3): m(x)
    if device.type == "cuda": torch.cuda.synchronize()
    t = time.time()
    for _ in range(runs): m(x)
    if device.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / runs

results = {}
for tname, ds in test_sets.items():
    ld = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    results[tname] = {}
    for label, m in models.items():
        r = evaluate(m, ld); r["sec_per_image"] = time_model(m); r["params_M"] = count_params(m) / 1e6
        results[tname][label] = r
print(f"{'set':10s} {'model':30s} {'PSNR':>7s} {'SSIM':>8s} {'s/img':>7s}")
for tname in results:
    for label, r in results[tname].items():
        print(f"{tname:10s} {label:30s} {r['psnr']:7.2f} {r['ssim']:8.4f} {r['sec_per_image']:7.3f}")
json.dump(results, open(OUTPUT_ROOT / "results/metrics/test_metrics.json", "w"), indent=2)
print("\nSaved:", OUTPUT_ROOT / "results/metrics/test_metrics.json")

In [ ]:
# Cell 12 — Qualitative comparison grids (Input | Baseline | Proposed | Ground truth)
@torch.no_grad()
def grid(tname, n=6, start=0):
    ds = test_sets[tname]; cols = list(models.keys()) + ["Ground truth"]
    fig, axes = plt.subplots(n, len(cols), figsize=(3.2 * len(cols), 3.2 * n), squeeze=False)
    for r in range(n):
        it = ds[start + r]; x = it["input"][None].to(device)
        imgs = [m(x).float().clamp(0, 1)[0].cpu() for m in models.values()] + [it["target"]]
        for c, im in enumerate(imgs):
            axes[r, c].imshow(im.permute(1, 2, 0).numpy()); axes[r, c].axis("off")
            if r == 0: axes[r, c].set_title(cols[c], fontsize=9)
    plt.tight_layout(); out = OUTPUT_ROOT / "results/figures" / f"{tname}_comparison.png"
    plt.savefig(out, dpi=120, bbox_inches="tight"); plt.show(); print("saved", out)
grid("real"); grid("synthetic")

## What to report

* The table in Cell 11 is your final result. The **"Input (no processing)"** row is the reference: a model only removes flares if it scores **above** that row.
* Training curves (Cell 10) and the grids (Cell 12) are the figures for the report.
* Both models share the same pretrained encoder, data cache, loss and schedule, so the comparison is fair. If the proposed model does not beat the baseline, report that as a finding: attention gates and a multi-scale bottleneck do not always help under a short training budget.

**If you disconnect:** re-run Cells 1–4 (setup), Cell 5 only if `/kaggle/temp` was wiped (it skips existing files), then re-run Cell 8 / 9; they resume from the last finished epoch.